In [1]:
import matplotlib.pyplot as plt
import re
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.utils import to_categorical
from gensim.models import Word2Vec


texts = [
    # Positive
    "The weather is so Beautiful today!",
    "I had a Wonderful breakfast this morning.",
    "Feeling GREAT after my workout!",
    "Such a lovely day outside :)",

    # Neutral
    "I went to the store to buy some groceries.",
    "The meeting started at nine in the morning.",
    "She read a book for an hour.",
    "He walked to the office this morning.",

    # Negative
    "I am SO tired and exhausted...",
    "This day has been awful!!",
    "Feeling really DOWN today.",
    "Everything is going WRONG."
]

# 0 = Negative | 1 = Neutral | 2 = Positive
labels = np.array([2, 2, 2, 2,
                   1, 1, 1, 1,
                   0, 0, 0, 0])


# ──────────────────────────────────────────────
#  STEP 2: CLEAN THE DATA (done ONCE for all 3 methods)
#
#  We clean here so all 3 methods use the same
#  clean input — fair and consistent!
# ──────────────────────────────────────────────

def clean_text(text):
    text = text.lower()                    # "Beautiful" → "beautiful"
    text = re.sub(r'[^a-z\s]', '', text)  # remove !, ., :), numbers, etc.
    text = text.strip()                    # remove leading/trailing spaces
    return text

cleaned_texts = [clean_text(t) for t in texts]

print("=" * 55)
print("  ORIGINAL  vs  CLEANED")
print("=" * 55)
for original, cleaned in zip(texts, cleaned_texts):
    print(f"  Before : {original}")
    print(f"  After  : {cleaned}")
    print()




In [5]:
# ──────────────────────────────────────────────
#  STEP 3: FFNN BUILDER
#  (same simple structure as reference code)
# ──────────────────────────────────────────────

def build_ffnn(input_size):
    model = Sequential()
    model.add(Dense(8, activation='relu', input_shape=(input_size,)))
    model.add(Dense(3, activation='softmax'))   # 3 classes → softmax
    model.compile(optimizer='adam',
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model


def predict_sentiment(model, vector):
    probs = model.predict(vector, verbose=0)[0]
    classes = ["Negative", "Neutral", "Positive"]
    result = classes[np.argmax(probs)]
    print(f"   Neg: {probs[0]:.2f}  Neu: {probs[1]:.2f}  Pos: {probs[2]:.2f}  →  {result}")




In [6]:
# ══════════════════════════════════════════════
#  METHOD 1: BAG OF WORDS
#  "Count how many times each word appears"
# ══════════════════════════════════════════════

print("=" * 55)
print("  METHOD 1: BAG OF WORDS")
print("=" * 55)

bow_vectorizer = CountVectorizer()
X_bow = bow_vectorizer.fit_transform(cleaned_texts).toarray()

print("\nVocabulary:")
print(bow_vectorizer.get_feature_names_out())

print("\nBoW Matrix (each row = 1 sentence, values = word counts):")
print(X_bow)

# Build model
model_bow = build_ffnn(X_bow.shape[1])

# Model Summary
print("\n--- Model Summary ---")
model_bow.summary()

# Train
history_bow = model_bow.fit(X_bow, labels, epochs=100, verbose=0)

_, acc = model_bow.evaluate(X_bow, labels, verbose=0)
print(f"\nTraining Accuracy: {acc*100:.2f}%")

# Predict new text
new_text = ["i like the weather today"]
new_vec_bow = bow_vectorizer.transform(new_text).toarray()
print(f"\nPredicting: '{new_text[0]}'")
predict_sentiment(model_bow, new_vec_bow)

# ── Plots ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Bag of Words', fontsize=14, fontweight='bold')

# Plot 1 — Training Loss
axes[0].plot(history_bow.history['loss'], color='blue')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')

# Plot 2 — Actual vs Predicted
predictions_bow = np.argmax(model_bow.predict(X_bow, verbose=0), axis=1)
axes[1].scatter(range(len(labels)), labels, color='blue', label='Actual', zorder=2)
axes[1].scatter(range(len(predictions_bow)), predictions_bow, color='red', label='Predicted', marker='x', s=100, zorder=3)
axes[1].set_title('Actual vs Predicted')
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('Class (0=Neg, 1=Neu, 2=Pos)')
axes[1].legend()
axes[1].set_yticks([0, 1, 2])

plt.tight_layout()
plt.show()


In [7]:
# ══════════════════════════════════════════════
#  METHOD 2: TF-IDF
#  "Rare words get higher scores,
#   common words (the, is) get lower scores"
# ══════════════════════════════════════════════

print("\n" + "=" * 55)
print("  METHOD 2: TF-IDF")
print("=" * 55)

tfidf_vectorizer = TfidfVectorizer()
X_tfidf = tfidf_vectorizer.fit_transform(cleaned_texts).toarray()

print("\nVocabulary:")
print(tfidf_vectorizer.get_feature_names_out())

print("\nTF-IDF Matrix (values are weights, not counts):")
print(np.round(X_tfidf, 3))

# Build model
model_tfidf = build_ffnn(X_tfidf.shape[1])

# Model Summary
print("\n--- Model Summary ---")
model_tfidf.summary()

# Train
history_tfidf = model_tfidf.fit(X_tfidf, labels, epochs=100, verbose=0)

_, acc = model_tfidf.evaluate(X_tfidf, labels, verbose=0)
print(f"\nTraining Accuracy: {acc*100:.2f}%")

# Predict new text
new_vec_tfidf = tfidf_vectorizer.transform(new_text).toarray()
print(f"\nPredicting: '{new_text[0]}'")
predict_sentiment(model_tfidf, new_vec_tfidf)

# ── Plots ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('TF-IDF', fontsize=14, fontweight='bold')

# Plot 1 — Training Loss
axes[0].plot(history_tfidf.history['loss'], color='blue')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')

# Plot 2 — Actual vs Predicted
predictions_tfidf = np.argmax(model_tfidf.predict(X_tfidf, verbose=0), axis=1)
axes[1].scatter(range(len(labels)), labels, color='blue', label='Actual', zorder=2)
axes[1].scatter(range(len(predictions_tfidf)), predictions_tfidf, color='red', label='Predicted', marker='x', s=100, zorder=3)
axes[1].set_title('Actual vs Predicted')
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('Class (0=Neg, 1=Neu, 2=Pos)')
axes[1].legend()
axes[1].set_yticks([0, 1, 2])

plt.tight_layout()
plt.show()


In [8]:
# ══════════════════════════════════════════════
#  METHOD 3: WORD2VEC
#  "Turn each word into numbers that capture
#   its meaning, then average for the sentence"
# ══════════════════════════════════════════════

print("\n" + "=" * 55)
print("  METHOD 3: WORD2VEC")
print("=" * 55)

# Split cleaned sentences into word lists — Word2Vec needs this
tokenized = [sentence.split() for sentence in cleaned_texts]

print("\nTokenized sentences:")
for tok in tokenized:
    print(" ", tok)

# Train Word2Vec on our tokenized sentences
w2v_model = Word2Vec(sentences=tokenized,
                     vector_size=10,   # each word = 10 numbers
                     window=3,
                     min_count=1,
                     seed=42,
                     epochs=100)

print("\nWord2Vec Vocabulary:")
print(list(w2v_model.wv.key_to_index.keys()))

print("\nExample — vector for the word 'wonderful':")
print(np.round(w2v_model.wv['wonderful'], 4))

# Represent each sentence = average of its word vectors
def sentence_to_vector(words, model, dim=10):
    vecs = [model.wv[w] for w in words if w in model.wv]
    return np.mean(vecs, axis=0) if vecs else np.zeros(dim)

X_w2v = np.array([sentence_to_vector(tok, w2v_model) for tok in tokenized])

print("\nWord2Vec Matrix (each sentence = 10 averaged numbers):")
print(np.round(X_w2v, 4))

# Build model
model_w2v = build_ffnn(10)

# Model Summary
print("\n--- Model Summary ---")
model_w2v.summary()

# Train
history_w2v = model_w2v.fit(X_w2v, labels, epochs=100, verbose=0)

_, acc = model_w2v.evaluate(X_w2v, labels, verbose=0)
print(f"\nTraining Accuracy: {acc*100:.2f}%")

# Predict new text
new_vec_w2v = sentence_to_vector(new_text[0].split(), w2v_model).reshape(1, -1)
print(f"\nPredicting: '{new_text[0]}'")
predict_sentiment(model_w2v, new_vec_w2v)

# ── Plots ──
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.suptitle('Word2Vec', fontsize=14, fontweight='bold')

# Plot 1 — Training Loss
axes[0].plot(history_w2v.history['loss'], color='blue')
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Loss')

# Plot 2 — Actual vs Predicted
predictions_w2v = np.argmax(model_w2v.predict(X_w2v, verbose=0), axis=1)
axes[1].scatter(range(len(labels)), labels, color='blue', label='Actual', zorder=2)
axes[1].scatter(range(len(predictions_w2v)), predictions_w2v, color='red', label='Predicted', marker='x', s=100, zorder=3)
axes[1].set_title('Actual vs Predicted')
axes[1].set_xlabel('Sample Index')
axes[1].set_ylabel('Class (0=Neg, 1=Neu, 2=Pos)')
axes[1].legend()
axes[1].set_yticks([0, 1, 2])

plt.tight_layout()
plt.show()


In [9]:
# ══════════════════════════════════════════════
#  FINAL SUMMARY
# ══════════════════════════════════════════════

print("\n" + "=" * 55)
print("  ACCURACY SUMMARY")
print("=" * 55)
for name, model, X in [
    ("Bag of Words", model_bow,   X_bow),
    ("TF-IDF",       model_tfidf, X_tfidf),
    ("Word2Vec",     model_w2v,   X_w2v),
]:
    _, acc = model.evaluate(X, labels, verbose=0)
    print(f"  {name:<15} →  {acc*100:.2f}%")

print("""
======================================================
  KEY TAKEAWAYS
======================================================

  ✔ Data Cleaning
    Done ONCE before all methods.
    Lower case + remove punctuation = fair input for all.

  ✔ Bag of Words
    Counts word occurrences → sparse matrix of integers.
    Simple and fast. No understanding of word meaning.

  ✔ TF-IDF
    Weights words by importance across all documents.
    Rare unique words score higher than common ones.

  ✔ Word2Vec
    Learns word meaning as dense vectors (10 numbers).
    Sentence = average of its word vectors.
    Needs large data to work best.

  ✔ FFNN (same for all 3)
    Input → Dense(8, ReLU) → Dense(3, Softmax)
    3 output nodes = Negative / Neutral / Positive
======================================================
""")